# 19: A detector for current generators (E9)

E6 to E8 deliberately trained only on OpenFake's training split. That split leaves out the newest generators, so the experiments could measure how well a detector handles generators it has never seen. For the website the goal is different: it should work as well as possible on the images people actually upload today, such as images from ChatGPT (GPT Image 1.5 and 2), Gemini (Nano Banana Pro) and Grok (Aurora).

This notebook trains a **deployment detector** on everything we can spare:

| OpenFake file | Use |
|---|---|
| training files from E8 (about 17,800 images) | training |
| test files 4 to 12 (not used by any experiment) | training |
| test file 3 | threshold calibration (as in notebook 18) |
| test files 0 to 2 | testing: the same images as E6 |

The test generators are now **seen** during training, so the results measure detection of known generators on new images, not generalisation. E6 to E8 remain the generalisation results.

Training images are prepared the way the website sees them: up to four 224 x 224 tiles at original resolution from the tile grid, both as distributed and after a simulated WhatsApp share (long side at most 1600 pixels, JPEG quality 80, as measured in E6). The detector is the same as M2: frozen CLIP ViT-L/14 features and logistic regression.

GPU runtime needed; about 5 GB is downloaded. Total time about 45 minutes; features are saved to Drive after every file, so a restart resumes. The last section uploads the detector to Hugging Face and needs a **write** token, typed into a hidden prompt.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q huggingface_hub pillow-heif

import io
import json
import os
import time
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pillow_heif
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image, ImageOps
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score, roc_curve
from transformers import CLIPVisionModelWithProjection

pillow_heif.register_heif_opener()
ROOT = Path("/content/drive/MyDrive/minor-project")
RESULTS = ROOT / "results"
E8_DATA = ROOT / "processed_openfake_train_full"
DEPLOY = ROOT / "processed_deploy"
DEPLOY.mkdir(exist_ok=True)
DOWNLOADS = Path("/content/openfake_test")
OWN_TEST = ROOT / "own_test"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Settings

In [ ]:
TRAIN_FILES = list(range(4, 13))
VAL_FILE = 12                    # also used to choose the regularisation strength, then included in the final fit
CALIBRATION_FILE = 3
TEST_FILES = [0, 1, 2]
TILES_PER_IMAGE = 4              # training tiles per image and version
TEST_PER_SOURCE = 80             # test images per generator
TEST_PER_REAL_SOURCE = 200       # test images per real source (DOCCI, ImageNet)
CALIBRATION_PER_CLASS = 300
WA_LONG_SIDE = 1600              # WhatsApp behaviour measured in E6
WA_QUALITY = 80
SIZE = 224
MAX_TILES = 9
SEED = 42
REPO = "tanish-237/ai-image-forensics-models"
CLIP_NAME = "openai/clip-vit-large-patch14"


def test_file(i):
    return f"core/test-{i:05d}-of-00013.parquet"

## Helper functions

In [ ]:
STD_LUMA = np.array([
    16, 11, 10, 16, 24, 40, 51, 61,
    12, 12, 14, 19, 26, 58, 60, 55,
    14, 13, 16, 24, 40, 57, 69, 56,
    14, 17, 22, 29, 51, 87, 80, 62,
    18, 22, 37, 56, 68, 109, 103, 77,
    24, 35, 55, 64, 81, 104, 113, 92,
    49, 64, 78, 87, 103, 121, 120, 101,
    72, 92, 95, 98, 112, 100, 103, 99,
])


def estimate_quality(img):
    luma = np.array(img.quantization[0])
    best_quality, best_error = 0, np.inf
    for quality in range(1, 101):
        scale = 5000 // quality if quality < 50 else 200 - 2 * quality
        table = np.clip((STD_LUMA * scale + 50) // 100, 1, 255)
        error = np.abs(table - luma).sum()
        if error < best_error:
            best_quality, best_error = quality, error
    return best_quality


def center_crop_aligned(img, size=SIZE):
    width, height = img.size
    left = ((width - size) // 2) // 16 * 16
    top = ((height - size) // 2) // 16 * 16
    return img.crop((left, top, left + size, top + size))


def jpeg_roundtrip(img, quality):
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=int(quality))
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")


def share_like_whatsapp(img):
    """Simulated WhatsApp share: long side at most 1600 pixels, then JPEG quality 80."""
    w, h = img.size
    scale = min(1.0, WA_LONG_SIDE / max(w, h))
    if scale < 1:
        img = img.resize((round(w * scale), round(h * scale)), Image.BICUBIC)
    return jpeg_roundtrip(img, WA_QUALITY)


def tile_positions(width, height, size=SIZE, max_tiles=MAX_TILES):
    """Evenly spaced, 16-pixel-aligned tile corners covering the whole image (same grid as the website)."""
    per_side = int(np.sqrt(max_tiles))
    xs = np.unique(np.linspace(0, width - size, min(per_side, max(1, width // size))).astype(int) // 16 * 16)
    ys = np.unique(np.linspace(0, height - size, min(per_side, max(1, height // size))).astype(int) // 16 * 16)
    return [(x, y) for y in ys for x in xs]


def enlarge_if_small(img):
    if min(img.size) >= SIZE:
        return img
    factor = SIZE / min(img.size)
    return img.resize((max(SIZE, round(img.size[0] * factor)), max(SIZE, round(img.size[1] * factor))), Image.BICUBIC)


def tiles_of(img, positions=None):
    img = enlarge_if_small(img)
    positions = tile_positions(*img.size) if positions is None else positions
    return np.stack([np.asarray(img.crop((x, y, x + SIZE, y + SIZE))) for x, y in positions])

## CLIP feature extractor (frozen)

In [ ]:
dtype = torch.float16 if device.type == "cuda" else torch.float32
clip = CLIPVisionModelWithProjection.from_pretrained(CLIP_NAME, torch_dtype=dtype).to(device).eval()
CLIP_SIZE = clip.config.image_size
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)


@torch.no_grad()
def clip_features(arr):
    out = []
    for s in range(0, len(arr), 128):
        x = torch.from_numpy(np.array(arr[s:s + 128])).to(device).permute(0, 3, 1, 2).float() / 255
        if x.shape[-1] != CLIP_SIZE:
            x = F.interpolate(x, size=CLIP_SIZE, mode="bicubic", align_corners=False)
        emb = clip(pixel_values=((x - CLIP_MEAN) / CLIP_STD).to(dtype)).image_embeds.float()
        out.append(F.normalize(emb, dim=1).cpu().numpy())
    return np.concatenate(out)


def as_distributed(img, rng):
    """Bias control as in notebook 03: a non-JPEG image gets one JPEG compression at a quality drawn from the real images."""
    rgb = img.convert("RGB")
    return rgb if img.format == "JPEG" else jpeg_roundtrip(rgb, rng.choice(REAL_QUALITIES))


e8_meta = pd.read_csv(E8_DATA / "meta.csv")
REAL_QUALITIES = e8_meta.loc[(e8_meta["label"] == "real") & (e8_meta["split"] == "train"), "jpeg_quality"].dropna().astype(int).to_numpy()
print(f"{len(REAL_QUALITIES)} real JPEG qualities, median {np.median(REAL_QUALITIES):.0f}")

## Training features from test files 4 to 12 (about 30 minutes, only the first time)

For every usable image: up to four tiles from the website's tile grid, once as distributed and once after the simulated share. Only the CLIP features are kept.

In [ ]:
def training_features(index):
    out = DEPLOY / f"train_{index:02d}.npz"
    if out.exists():
        return
    start = time.time()
    path = hf_hub_download("ComplexDataLab/OpenFake", test_file(index), repo_type="dataset", local_dir=DOWNLOADS)
    pf = pq.ParquetFile(path)
    rng = np.random.default_rng(SEED + index)
    features, rows, image_id = [], [], 0
    for g in range(pf.num_row_groups):
        tiles = []
        for item in pf.read_row_group(g, columns=["image", "label", "model"]).to_pylist():
            try:
                img = Image.open(io.BytesIO(item["image"]["bytes"]))
                img.load()
            except Exception:
                continue
            if img.mode in ("L", "LA", "1", "I", "I;16") or min(img.size) < SIZE:
                continue
            for version, view in [("original", as_distributed(img, rng)), ("shared", share_like_whatsapp(img.convert("RGB")))]:
                view = enlarge_if_small(view)
                grid = tile_positions(*view.size)
                picked = [grid[k] for k in rng.choice(len(grid), size=min(TILES_PER_IMAGE, len(grid)), replace=False)]
                tiles.append(tiles_of(view, picked))
                rows += [{"image": image_id, "source": item["model"], "y": int(item["label"] == "fake"), "version": version}] * len(picked)
            image_id += 1
        if tiles:
            features.append(clip_features(np.concatenate(tiles)))
    meta = pd.DataFrame(rows)
    np.savez(out, features=np.concatenate(features).astype(np.float16), y=meta["y"].to_numpy(), image=meta["image"].to_numpy(),
             source=meta["source"].to_numpy().astype(str), version=meta["version"].to_numpy().astype(str))
    os.remove(path)
    print(f"test file {index}: {image_id} images, {len(meta)} tiles, {meta.groupby('image')['y'].first().mean():.0%} generated "
          f"({time.time() - start:.0f}s)")


for index in TRAIN_FILES:
    training_features(index)

parts = {i: np.load(DEPLOY / f"train_{i:02d}.npz") for i in TRAIN_FILES}
new_images = pd.concat([pd.DataFrame({"file": i, "image": p["image"], "source": p["source"]}).drop_duplicates() for i, p in parts.items()])
print(f"New training images: {len(new_images)}")
display(new_images["source"].value_counts().to_frame("images"))

## Fit the deployment detector

The regularisation strength is chosen on test file 12 plus the E8 validation file, then the final classifier is fitted on all training data. Class weights are balanced because the test files contain more generated than real images.

In [ ]:
e8_y = (e8_meta["label"] == "fake").astype(int).to_numpy()
e8_train_x, e8_train_y = np.load(E8_DATA / "clip_train.npy"), e8_y[(e8_meta["split"] == "train").to_numpy()]
e8_val_x, e8_val_y = np.load(E8_DATA / "clip_val.npy"), e8_y[(e8_meta["split"] == "val").to_numpy()]


def stack(indices):
    return (np.concatenate([parts[i]["features"].astype(np.float32) for i in indices]),
            np.concatenate([parts[i]["y"] for i in indices]))


fit_x, fit_y = stack([i for i in TRAIN_FILES if i != VAL_FILE])
val_x, val_y = stack([VAL_FILE])
fit_x, fit_y = np.concatenate([e8_train_x, fit_x]), np.concatenate([e8_train_y, fit_y])
val_x, val_y = np.concatenate([e8_val_x, val_x]), np.concatenate([e8_val_y, val_y])
print(f"choosing C on {len(fit_y)} training and {len(val_y)} validation tiles")

search = []
for C in [0.1, 1, 10, 100]:
    c = LogisticRegression(C=C, max_iter=5000, class_weight="balanced").fit(fit_x, fit_y)
    p = c.predict_proba(val_x)[:, 1]
    search.append((float(log_loss(val_y, p)), C))
    print(f"C={C}: validation loss {search[-1][0]:.4f}, AUC {roc_auc_score(val_y, p):.4f}")
C = min(search)[1]

all_x, all_y = np.concatenate([fit_x, val_x]), np.concatenate([fit_y, val_y])
deploy = LogisticRegression(C=C, max_iter=5000, class_weight="balanced").fit(all_x, all_y)
print(f"Deployment detector: C = {C}, fitted on {len(all_y)} tiles")

old_lr = np.load(hf_hub_download(REPO, "m2_lr.npz"))
old_config = json.loads(Path(hf_hub_download(REPO, "config.json")).read_text())
detectors = {
    "website before (E8)": (old_lr["coef"].reshape(-1), float(old_lr["intercept"])),
    "deployment (E9)": (deploy.coef_[0], float(deploy.intercept_[0])),
}
thresholds = {"website before (E8)": old_config["thresholds"]["M2"]}


def full_image_scores(img):
    """Average tile score of every detector, as on the website."""
    f = clip_features(tiles_of(img))
    return {name: float((1 / (1 + np.exp(-(f @ w + b)))).mean()) for name, (w, b) in detectors.items()}

## Calibrate the threshold on test file 3

As in notebook 18: 300 real and 300 generated images, full-image scores, the threshold with the best balanced accuracy. This time both versions of every image (as distributed and after the simulated share) are used, so one threshold fits both.

In [ ]:
def sample_rows(path, per_label=None, per_source=None, per_real_source=None):
    labels_table = pq.ParquetFile(path).read(columns=["label", "model"]).to_pandas()
    rng = np.random.default_rng(SEED)
    if per_label:
        groups = labels_table.groupby("label")
        limit = lambda name, n: min(n, per_label * 2)
    else:
        groups = labels_table.groupby("model")
        limit = lambda name, n: min(n, per_real_source if labels_table.loc[labels_table["model"] == name, "label"].iat[0] == "real" else per_source)
    chosen = set()
    for name, group in groups:
        chosen |= set(rng.choice(group.index.to_numpy(), size=limit(name, len(group)), replace=False).tolist())
    return chosen


def score_file(index, pick, cap_per_label=None):
    path = hf_hub_download("ComplexDataLab/OpenFake", test_file(index), repo_type="dataset", local_dir=DOWNLOADS)
    pf = pq.ParquetFile(path)
    chosen = pick(path)
    rows, start, counts = [], 0, {"real": 0, "fake": 0}
    for g in range(pf.num_row_groups):
        n = pf.metadata.row_group(g).num_rows
        wanted = [r for r in range(start, start + n) if r in chosen]
        if wanted:
            t = pf.read_row_group(g, columns=["image", "label", "model"]).to_pylist()
            for r in wanted:
                item = t[r - start]
                img = Image.open(io.BytesIO(item["image"]["bytes"]))
                if img.mode in ("L", "LA", "1", "I", "I;16") or min(img.size) < SIZE:
                    continue
                if cap_per_label and counts[item["label"]] >= cap_per_label:
                    continue
                counts[item["label"]] += 1
                rgb = img.convert("RGB")
                base = {"file": index, "source": item["model"], "format": img.format, "y": int(item["label"] == "fake")}
                rows.append({**base, "version": "original", **full_image_scores(rgb)})
                rows.append({**base, "version": "shared", **full_image_scores(share_like_whatsapp(rgb))})
        start += n
    os.remove(path)
    return pd.DataFrame(rows)


calibration = score_file(CALIBRATION_FILE, lambda p: sample_rows(p, per_label=CALIBRATION_PER_CLASS), CALIBRATION_PER_CLASS)
calibration.to_csv(RESULTS / "e9_calibration.csv", index=False)
fpr, tpr, thr = roc_curve(calibration["y"], calibration["deployment (E9)"])
finite = np.isfinite(thr)
fpr, tpr, thr = fpr[finite], tpr[finite], thr[finite]
best = np.argmax(tpr - fpr)
thresholds["deployment (E9)"] = float(np.clip(thr[best], 0.01, 0.99))
for name in detectors:
    print(f"{name}: calibration AUC {roc_auc_score(calibration['y'], calibration[name]):.3f}, threshold {thresholds[name]:.3f}")

## Test on files 0 to 2

Up to 80 images per generator and 200 per real source, full image, as distributed and after the simulated share. Each detector uses its own calibrated threshold.

In [ ]:
test_path = RESULTS / "e9_test_scores.csv"
if test_path.exists():
    test = pd.read_csv(test_path)
else:
    test = pd.concat([score_file(i, lambda p: sample_rows(p, per_source=TEST_PER_SOURCE, per_real_source=TEST_PER_REAL_SOURCE)) for i in TEST_FILES],
                     ignore_index=True)
    test.to_csv(test_path, index=False)


def summarise(frame, group=None):
    rows = []
    for name in detectors:
        decided = frame[name].to_numpy() >= thresholds[name]
        y = frame["y"].to_numpy()
        row = {"detector": name, "real_called_real": float((~decided[y == 0]).mean()) if (y == 0).any() else np.nan,
               "ai_detected": float(decided[y == 1].mean()) if (y == 1).any() else np.nan}
        if len(set(y)) == 2:
            row["auc"] = float(roc_auc_score(y, frame[name]))
        rows.append(row)
    return pd.DataFrame(rows)


summary = pd.concat([summarise(part).assign(version=version) for version, part in test.groupby("version")], ignore_index=True)
summary.to_csv(RESULTS / "e9_summary.csv", index=False)
display(summary.round(3))

correct = test[["source", "version", "y"]].copy()
for name in detectors:
    correct[name] = (test[name] >= thresholds[name]).astype(int) == test["y"]
per_source = correct.groupby(["source", "version"])[list(detectors)].mean().unstack("version")
per_source["images"] = test[test["version"] == "original"].groupby("source").size()
per_source.to_csv(RESULTS / "e9_per_source.csv")
display(per_source.round(3))

## Real WhatsApp images (E6) and our own images

The 100 image pairs from E6 come from test files 0 to 2, so they were not used for training. Our own images (photos taken with our phones and images we generated, never published) are read from `own_test/real/` and `own_test/ai/` on Drive if that folder exists. Only the pixels are used here; the file's metadata plays no part.

In [ ]:
pairs = pd.read_csv(RESULTS / "e6_whatsapp_predictions.csv")
rows = []
for when, folder, column in [("before", ROOT / "whatsapp" / "original", "file"), ("after", ROOT / "whatsapp" / "received", "received_file")]:
    scores = pd.DataFrame([full_image_scores(Image.open(folder / f).convert("RGB")) for f in pairs[column]])
    scores["y"] = pairs["y"].to_numpy()
    rows.append(summarise(scores).assign(when=when))
whatsapp = pd.concat(rows, ignore_index=True)
whatsapp.to_csv(RESULTS / "e9_whatsapp.csv", index=False)
display(whatsapp.round(3))

own = []
for label, y in [("real", 0), ("ai", 1)]:
    folder = OWN_TEST / label
    if not folder.exists():
        continue
    for f in sorted(folder.iterdir()):
        try:
            img = ImageOps.exif_transpose(Image.open(f)).convert("RGB")
        except Exception:
            continue
        scores = full_image_scores(img)
        own.append({"file": f.name, "y": y, "size": f"{img.size[0]}x{img.size[1]}",
                    **{name: s for name, s in scores.items()},
                    **{f"{name} correct": (s >= thresholds[name]) == bool(y) for name, s in scores.items()}})
if own:
    own = pd.DataFrame(own)
    own.to_csv(RESULTS / "e9_own_test.csv", index=False)
    display(own.round(3))
    display(summarise(own).round(3))
else:
    print("No own_test folder on Drive; skipped.")

## Chart

In [ ]:
order = per_source.sort_values(("deployment (E9)", "shared"))
colors = {"website before (E8)": "#a9b4c2", "deployment (E9)": "#2a78d6"}
fig, axes = plt.subplots(1, 2, figsize=(14, 0.32 * len(order) + 1.8), sharey=True)
y_pos = np.arange(len(order))
for ax, version, title in zip(axes, ["original", "shared"], ["As distributed", "After simulated WhatsApp share"]):
    for k, name in enumerate(detectors):
        ax.barh(y_pos + (k - 0.5) * 0.4, order[(name, version)] * 100, 0.38, color=colors[name], label=name)
    ax.set_title(title, fontsize=11)
    ax.set_xlim(0, 100)
    ax.axvline(50, color="#999999", linewidth=1, linestyle="--")
    ax.grid(axis="x", color="#e5e5e5")
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel("Correct decisions (%)")
axes[0].set_yticks(y_pos, order.index)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(RESULTS / "e9_deploy.png", dpi=120)
plt.show()

## Upload to Hugging Face

Adds `clip_lr_deploy.npz` and a `detector` entry in `config.json`; the website uses it as soon as it is restarted. The E8 files stay in the repository.

In [ ]:
OUT = Path("/content/deploy_upload")
OUT.mkdir(exist_ok=True)
np.savez(OUT / "clip_lr_deploy.npz", coef=deploy.coef_.astype(np.float32), intercept=np.array(deploy.intercept_[0], dtype=np.float32))
old_config["detector"] = {
    "file": "clip_lr_deploy.npz",
    "threshold": thresholds["deployment (E9)"],
    "name": "CLIP + linear classifier, trained on OpenFake including GPT Image, Nano Banana Pro and Aurora",
}
old_config["description"] = "E9 deployment detector"
(OUT / "config.json").write_text(json.dumps(old_config, indent=2))
print(json.dumps(old_config["detector"], indent=2))

api = HfApi(token=getpass("Hugging Face write token: "))
api.upload_folder(folder_path=OUT, repo_id=REPO, repo_type="model", commit_message="Add E9 deployment detector")
print(f"Uploaded to https://huggingface.co/{REPO}")